In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

In [15]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])

train_dataset = datasets.MNIST(root="./data", transform=transform, train=True, download=True)
test_dataset = datasets.MNIST(root="./data", transform=transform, train=False, download=True)

train_dataloader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=64)
test_dataloader = DataLoader(dataset=test_dataset, shuffle=False, batch_size=64)

In [16]:
class MNIST(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d( # 28x28
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(2), # 14x14

        nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        ),
        nn.MaxPool2d(2),
    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )
  def forward(self, x):
    x = self.conv(x)
    x = self.fc(x)
    return x

In [17]:
model = MNIST()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [20]:
epoches = 5

model.train()
for epoch in range(epoches):
  running_loss = 0.0
  for images, labels in train_dataloader:
    optimizer.zero_grad()

    prediction = model(images)
    loss = criterion(prediction, labels)

    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  epoch_loss = running_loss / len(train_dataloader)
  print(f"Epoch: {epoch + 1}, loss: {loss.item():.4f}")
  torch.save(model.state_dict(), f"mnist{epoch + 1}.pth")

Epoch: 1, loss: 0.1221
Epoch: 2, loss: 0.0733
Epoch: 3, loss: 0.0170
Epoch: 4, loss: 0.0603
Epoch: 5, loss: 0.1990


In [26]:
model.load_state_dict(torch.load("mnist4_.pth"))

model.eval()

correct, total = 0, 0

with torch.no_grad():
  for images, labels in test_dataloader:
    outputs = model(images)
    predictions = outputs.argmax(dim=1)

    correct += (predictions == labels).sum().item()
    total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 98.89%
